### Add new document to the ChromaDB

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

import numpy as np


In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_core.prompts import ChatPromptTemplate   



In [8]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-3.5-turbo"
)


In [9]:
#  If your ChromaDB was created with persistence, you can connect to it like this

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vectordb = Chroma(
    persist_directory="./chroma_rag_db",
    embedding_function=embeddings,
    collection_name="kcj-langchain-rag-chromadb"
)

retriever = vectordb.as_retriever()

In [11]:
new_document = """
Blockchain:
A blockchain is a decentralized digital ledger that records transactions across multiple computers in such a way that the registered transactions cannot be altered retroactively. 
This technology underpins cryptocurrencies like Bitcoin and Ethereum, but its applications extend far beyond digital currencies. 
Blockchains are used for secure and transparent record-keeping in various industries, including supply chain management, healthcare, finance, and voting systems.
"""
new_document


'\nBlockchain:\nA blockchain is a decentralized digital ledger that records transactions across multiple computers in such a way that the registered transactions cannot be altered retroactively. \nThis technology underpins cryptocurrencies like Bitcoin and Ethereum, but its applications extend far beyond digital currencies. \nBlockchains are used for secure and transparent record-keeping in various industries, including supply chain management, healthcare, finance, and voting systems.\n'

In [12]:
from langchain.schema import Document
new_document = Document(
    page_content=new_document,
    metadata={"source": "additional_doc.txt", "topic": "Blockchain"}
)
new_document

Document(metadata={'source': 'additional_doc.txt', 'topic': 'Blockchain'}, page_content='\nBlockchain:\nA blockchain is a decentralized digital ledger that records transactions across multiple computers in such a way that the registered transactions cannot be altered retroactively. \nThis technology underpins cryptocurrencies like Bitcoin and Ethereum, but its applications extend far beyond digital currencies. \nBlockchains are used for secure and transparent record-keeping in various industries, including supply chain management, healthcare, finance, and voting systems.\n')

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
new_chunks = text_splitter.split_documents([new_document])
vectordb.add_documents(new_chunks)

['66cad83f-9416-4c68-8967-1cafe6d7547d']

In [18]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [19]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

 
rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F2811C1BB0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001F29BDECAD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F29BE2AE70>, root_client

In [22]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [23]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What is BlockChain?")

Testing LCEL Chain:
Question: What is BlockChain?
--------------------------------------------------
Answer: Blockchain is a decentralized digital ledger that records transactions across multiple computers in such a way that the registered transactions cannot be altered retroactively. It underpins cryptocurrencies like Bitcoin and Ethereum and is used for secure and transparent record-keeping in various industries.

Source Documents:

--- Source 1 ---
Blockchain:
A blockchain is a decentralized digital ledger that records transactions across multiple computers in such a way that the registered transactions cannot be altered retroactively. 
This tec...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artifi